## 11.5 הולכת שגיאות: אנליטית ומונטה-קרלו

יש לנו `v0_measured` עם שגיאת מדידה (ה-SEM שחישבנו בסעיף 11.2). אבל מה שמעניין אותנו הוא לא `v0` עצמו -- זה הטווח התאורטי `R = v0**2/g * sin(2*theta)`. אם `v0` לא מדויק, גם `R` לא מדויק. השאלה: **כמה** לא מדויק?

שתי דרכים לענות:

1. **אנליטית** -- נוסחת הולכת שגיאות: אם `R = f(v0)`, אז `sigma_R ≈ |dR/dv0| * sigma_v0`.
2. **מונטה-קרלו** -- דוגמים הרבה ערכי `v0` מהתפלגות נורמלית סביב הממוצע עם סטיית התקן `sigma_v0`, מחשבים `R` לכל אחד, ומסתכלים על הפיזור של תוצאות ה-`R`.

שתי הדרכים אמורות להסכים -- וזו בדיוק הבדיקה שנעשה.

In [ ]:
import numpy as np
import pandas as pd

g = 9.8
df = pd.read_csv("lab_measurements.csv")
df_clean = df.dropna()

angle_deg = 45.0
theta = np.radians(angle_deg)
sub = df_clean[df_clean["angle_deg"] == angle_deg]

v0_mean = sub["v0_measured"].mean()
n = len(sub)
v0_sem = sub["v0_measured"].std(ddof=1) / np.sqrt(n)

R_mean = v0_mean**2 / g * np.sin(2*theta)
print(f"v0 = {v0_mean:.3f} +/- {v0_sem:.4f} m/s  (n={n})")
print(f"R  = {R_mean:.3f} m")

### דרך 1: הולכת שגיאות אנליטית

הנגזרת: `dR/dv0 = 2*v0/g * sin(2*theta)`. מכפילים בערך המוחלט שלה ב-`sigma_v0`.

In [ ]:
dR_dv0 = 2 * v0_mean / g * np.sin(2*theta)
sigma_R_analytic = abs(dR_dv0) * v0_sem
print(f"sigma_R (אנליטי) = {sigma_R_analytic:.4f} m")

### דרך 2: מונטה-קרלו

`rng.normal(v0_mean, v0_sem, size=N)` -- דוגמים N ערכי v0 "מדומים", כאילו מדדנו את v0 עוד ועוד פעמים. לכל אחד מחשבים R, ואז לוקחים `std` על כל תוצאות ה-R.

In [ ]:
rng = np.random.default_rng(42)
N = 20000
v0_samples = rng.normal(v0_mean, v0_sem, size=N)
R_samples = v0_samples**2 / g * np.sin(2*theta)

sigma_R_mc = R_samples.std(ddof=1)
print(f"sigma_R (מונטה-קרלו) = {sigma_R_mc:.4f} m")
print(f"היחס בין השתיים: {sigma_R_mc / sigma_R_analytic:.3f}")

שימו לב: היחס קרוב מאוד ל-1. זה לא מקרי -- כש-`sigma_v0` קטן יחסית ל-`v0` (כאן פחות מ-2%), `R(v0)` כמעט ליניארי בסביבת `v0_mean`, ולכן הקירוב האנליטי (מבוסס נגזרת -- כלומר קירוב ליניארי מקומי) קרוב מאוד לתוצאה המלאה של מונטה-קרלו, שלא מניחה שום דבר על צורת התלות.

### באג נפוץ: לשכוח את הערך המוחלט, או להשתמש ב-sigma_v0 בריבוע

שתי טעויות שכיחות בהולכת שגיאות:
1. `sigma_R = dR_dv0 * v0_sem` בלי `abs(...)` -- אם הנגזרת שלילית, מקבלים "שגיאה" שלילית, שאין לה משמעות פיזיקלית (וזה עובר בשקט -- שום שגיאה לא נזרקת).
2. בלבול בין `sigma_v0` (סטיית תקן) לבין `sigma_v0**2` (שונות) בנוסחה -- הכפלה בגודל הלא נכון נותנת תוצאה שגויה בסדר גודל, בלי אזהרה.

In [ ]:
dR_dv0_test = -5.0  # לצורך ההדגמה, נגזרת "שלילית" מדומה
sigma_v0_test = 0.3

wrong_no_abs = dR_dv0_test * sigma_v0_test
correct_with_abs = abs(dR_dv0_test) * sigma_v0_test
print(f"בלי abs: {wrong_no_abs:.3f}  (שגיאה שלילית - חסר משמעות)")
print(f"עם abs:  {correct_with_abs:.3f}")

### נסו בעצמכם

חשבו את `sigma_R` (אנליטי ומונטה-קרלו) עבור זווית 30 מעלות, וודאו שהיחס בין שתי השיטות קרוב ל-1.

In [ ]:
# angle_deg2 = 30.0
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
angle_deg2 = 30.0
theta2 = np.radians(angle_deg2)
sub2 = df_clean[df_clean["angle_deg"] == angle_deg2]

v0_mean2 = sub2["v0_measured"].mean()
v0_sem2 = sub2["v0_measured"].std(ddof=1) / np.sqrt(len(sub2))

dR_dv0_2 = 2 * v0_mean2 / g * np.sin(2*theta2)
sigma_R_analytic2 = abs(dR_dv0_2) * v0_sem2

R_samples2 = rng.normal(v0_mean2, v0_sem2, size=N)**2 / g * np.sin(2*theta2)
sigma_R_mc2 = R_samples2.std(ddof=1)

print(f"אנליטי: {sigma_R_analytic2:.4f}   מונטה-קרלו: {sigma_R_mc2:.4f}")
print(f"יחס: {sigma_R_mc2/sigma_R_analytic2:.3f}")
```
`````

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "מתי הקירוב האנליטי להולכת שגיאות (מבוסס נגזרת) פחות אמין?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "כשהשגיאה היחסית ב-v0 גדולה, כך שהפונקציה כבר לא כמעט-ליניארית בסביבת המדידה", "correct": True, "feedback": "נכון - קירוב הנגזרת מניח תלות ליניארית מקומית."},
            {"answer": "תמיד - מונטה-קרלו תמיד מדויק יותר ויש להעדיף אותו בכל מקרה", "correct": False, "feedback": "לא בהכרח - כששגיאת המדידה קטנה יחסית, שתי השיטות מסכימות היטב."},
            {"answer": "רק כשמשתמשים ב-NumPy", "correct": False, "feedback": "לא קשור לספרייה."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

חזרו על החישוב (אנליטי + מונטה-קרלו) עבור **כל חמש הזוויות** בלולאה (או ב-list comprehension), ואספו את תוצאות `sigma_R` לרשימה/מערך אחד לכל שיטה. וודאו שכל חמשת היחסים קרובים ל-1.

In [ ]:
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
angles_all = sorted(df_clean["angle_deg"].unique())
sigma_R_analytic_all = []
sigma_R_mc_all = []

for ang in angles_all:
    th = np.radians(ang)
    s = df_clean[df_clean["angle_deg"] == ang]
    v0m = s["v0_measured"].mean()
    v0sem = s["v0_measured"].std(ddof=1) / np.sqrt(len(s))

    dR = 2 * v0m / g * np.sin(2*th)
    sigma_R_analytic_all.append(abs(dR) * v0sem)

    R_s = rng.normal(v0m, v0sem, size=N)**2 / g * np.sin(2*th)
    sigma_R_mc_all.append(R_s.std(ddof=1))

sigma_R_analytic_all = np.array(sigma_R_analytic_all)
sigma_R_mc_all = np.array(sigma_R_mc_all)
print(np.round(sigma_R_mc_all / sigma_R_analytic_all, 3))
```
`````